# 基础材料准备 pipeline

原始资料 → 字节检查与图片索引复用 → 概念与视觉支持审核 → 独立视觉材料发布 → 文章联合提炼与终审。

文章与视觉材料是两个输出，文章未采用的图也可单独发布。三层信息分别为图片索引、概念支持范围、下游任务角色。

**本册统一编排文本整理与图片标注，包含视觉审核分支。** 视觉审核与发布的具体编排已移到独立视觉 notebook，本册调用同一实现；单独补视觉材料请运行 preparation/visual_materials_debug.ipynb，不必重跑文章。


文本与视觉基础材料。已有预标注按 SHA 只读复用并冻结；缺失索引由现有看图调用补齐。视觉发布不依赖文章选图，文章终审失败不使已发布视觉材料失效。任务角色由下游确定。已有独立视觉 V2 全量结果；本册不自动启动重跑。
视觉分支可单独运行：[preparation/visual_materials_debug.ipynb](visual_materials_debug.ipynb)。基础材料准备流程调用其中同一个审核与发布子图。


标注与审核统一位于 `curated/images.lance`：`descriptions[]` 是图片描述，`concept_matches[]` 是概念匹配，`concept_assessments[]` 是视觉审核；每行一个 SHA。提示词与参数在 `curation/preparation/configs/image_annotation_v2.json`，`config_id` 关联配置摘要。机器描述不代表知识或目标审核通过。
复用时给 `model_config.image_annotations_ref` 传 `curated/images.lance` 的固定 DatasetRef；多份有效描述用 `image_annotation_config_id` 明确选择；不按文件存在与否自动回退。`_demiflow/<表名>/checkpoint.json` 是平台提交回执，`write.lock` 是并发锁，二者不是图片标注内容。


## 当前材料表怎么读

原始图片来源是单张 `raw/images.lance`，一行一个 SHA；文档来源是单张 `raw/documents.lance`，一行一个确定版本。每种材料只传一个固定 DatasetRef，内部 fragment 由 Lance 管理。`source_scope=all` 的 QID 映射来自文档自身的 QID/语言/page_id，不依赖另一张映射表。

| 字段 | 含义 |
| --- | --- |
| `concepts` | 来源关联的概念去重列表，便于筛选；不等于审核通过 |
| `sources` | 每条采集来源的概念、查询词、URL、图注、版权与时间；不同来源不会相互覆盖 |
| `availability` | 图片已有 Lance Blob 或只有 metadata；没有字节会明确报告 |
| `resolution` | 实测显示宽高、存储宽高、MP、宽高比和测量来源；null 表示未测量 |
| `sources[].declared_width/height` | 来源声明的尺寸，不能当作实测尺寸 |
| `document_id / content_sha256` | 文档版本身份 / 正文摘要；不同版本独立保留 |
| `text / sections / images` | 正文或 typed 章节、文内图片引用，不把正文和图片 Blob 藏进来源 JSON |

图片客观索引、概念支持审核、任务参考/目标角色仍是三种不同结论，分别保留协议和引用。文章交付携带 `published_passages` 和 `published_images`，后者是最终配图 ID 到原图 SHA/来源的精确绑定；`images` 是原始候选池，不能因在池中就成为答案参考。

当前发布：原始材料 `material_entities_20260921`、知识文章 `knowledge_articles_20260921`、视觉材料 `visual_images_20260921`。分辨率未全库补跑；实际用图时测量。

原始图片与策展图片是两张表：`raw/images.lance` 保存字节和采集来源，`curated/images.lance` 保存标签与审核。`source_refs[]` 表示处理所用的固定原始 DatasetRef；补标签只更新 curated，采集更新 raw 不会自动改变已有发布。复用描述需要单独传入 curated 的固定引用。


**执行前重启内核，修改代码或提示词后使用新 RUN。**

默认只读查看已固定的交付；当前文章交付为 111 个机器审核通过概念，独立视觉已完成 V2 全量。后续新模型运行须使用新 run；正式 V2 训练数据与训练验证尚未执行。


### 当前正式执行链
原始 datasets → 文档清洗/过滤/去重、图片字节检查与 SHA 索引复用 → 概念身份及正文相关性 → **Qwen 图片初筛 → Gemma 独立复核含 keep 图的完整原批次 → 分歧暂缓** → PublishVisualMaterials 独立发布 → 原生图文关联与相似度补充 → 按容量组装 → `joint_paragraphs` → `final_review` → 程序校验与保存。

联合提取仅收到概念范围、编号原文及必要上下文、真实图片；最后 review 收到各组待审提取正文，以及提取稿实际引用的全部原文和采用的全部图片；提取稿中的断言必须回查原文。上游模型 caption 和筛选结论不作为证据。

两模型共用两张 GPU：阶段内部为 demiflow 流式处理，图片初筛和复核之间有完整 checkpoint 与服务切换。Gemma 完成后恢复原 Qwen 和预标注。`image_review_service='borrow'`管理借用恢复，`external`使用已准备好的外部服务。双模型一致不代表身份已经证实。


### 主线算子：作用、输入、输出

|算子|输入与作用|
|---|---|
|read_records → SelectSourceRecords → Concept/Document/ImageFromRecord|原始采集清单；按概念先过滤，再转换并保留来源记录|
|SelectConcept 与 join|选择概念、关联原始文档与图片|
|ReadDocument → CleanDocument → FilterDocumentBlocks|读取原文，清洗正文、过滤重复与导航，保留原文位置及必要上下文|
|CheckImage → ReuseImageAnnotations|检查字节、尺寸；按 SHA 冻结旧索引，缺失明确保留|
|PrepareIdentity → identity → ApplyIdentity → EnsureConceptLabel|检查材料身份；保留预览范围及未检查材料，统一概念分组名称|
|BuildSourceBlocks → select_blocks → ApplyBlockSelection|逐块判断对理解目标概念的实际作用，保留原文|
|SelectAvailableImages → select_images(Qwen/Gemma) → ApplyConfirmedImageSelection|独立查看真实像素；独立核验概念、可见支持并补齐缺失索引；显式发布视觉材料|
|PublishVisualMaterials|独立发布双模型通过的图及概念支持范围，不依赖文章选图|
|PrepareRoutingMaterials → EmbedParagraphBatch / EncodeImageTextMaterials|整理原生图文关系，计算分组所需向量|
|RouteByTokenBudget → BuildRoutedJointRequest|按实际文本和图片容量组装，保留完整材料覆盖|
|PrepareArticleInput → joint_paragraphs → ApplyArticle|简明原文与像素输入；提取普通文章，程序解析简单引用与配图标记|
|PrepareFinalReview → final_review → ApplyArticle(final=True)|汇集各组待审正文及全部实际使用的原文与像素，一次审查修订并写正文、选择图号|
|PublishArticle → checkpoint|程序附回来源、原图及统一图号，正文与配图分区保存；不生成图注，完整状态保留|

当前提示词：`identity.yaml`、`relevance.yaml`、`select_images.yaml`、`article_joint.yaml`、`final_review.yaml`。`joint_paragraphs`是调用名称，实际文章版模板为`article_joint.yaml`。state 只保存冻结副本和运行产物。

最终 review 同时检查来源、对象范围、冲突、重复和配图对应；程序检查完整性及引用范围。二者不能代替独立事实认证。无材料、无可靠知识和处理失败分别保存，失败稿不会直接发布。


## 1．处理与查看配置
MAX_RECORDS默认None：每个指定文件不限制扫描条数；填写整数才截断。IDS、材料分批、每次模型材料预算和调用预算是独立配置，保持显式。默认view_saved只查看已有最终文件，不启动全量扫描。


In [ ]:
from curation.preparation.stages import EncodeStage, stage_uri, read_stage, from_stage_row, PIPELINE_STAGE_ROWS
from pathlib import Path
import sys, asyncio, itertools, random, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# 不对运行中的Dataset热替换算子：发现旧内核时停止，重启后再读取checkpoint。
def _assert_current_kernel():
    import os
    try:
        shell = get_ipython()
    except NameError:
        return  # CLI是独立Python进程。
    if shell is None or not hasattr(shell, 'kernel'):
        return
    boot = next(int(line.split()[1]) for line in Path('/proc/stat').read_text().splitlines() if line.startswith('btime '))
    started = boot + int(Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    changed = []
    for name, module in tuple(sys.modules.items()):
        if name.startswith(('curation.', 'demiflow.')):
            filename = getattr(module, '__file__', None)
            if filename and Path(filename).is_file() and Path(filename).stat().st_mtime > started:
                changed.append(name)
    if changed:
        raise RuntimeError('当前内核启动后算子代码已更新，请重启内核并重新执行初始化；禁止新旧算子混用。涉及：' + ', '.join(changed[:6]))
_assert_current_kernel()
from curation.preparation.ops.filter_document_blocks import FilterDocumentBlocks
from curation.preparation.ops.select_source_records import SelectSourceRecords
from curation.preparation.ops.image_selection import SelectAvailableImages, SelectRelatedMaterials
from demiflow.standalone import local_data
from curation.preparation.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.preparation.ops.paragraph_similarity import EmbedParagraphBatch
from curation.preparation.ops.token_routing import RouteByTokenBudget
# 业务算子：只处理数据，不隐藏上/下游Dataset。
from curation.preparation.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks, ReadDocument, CleanDocument, CheckImage,
    CountMaterial, NestMaterial, merge_concept, merge_document, distinct,
    fill_material_counts, model_input)
from curation.preparation.ops.prompt_operators import PrepareIdentity, ApplyIdentity
from curation.preparation.ops.source_blocks import BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection, merge_block_decisions
from curation.preparation.ops.prompt_config import material_prompt_pack, prompt_execution_options, save_prompt_config
# 下面仅为原文件发现、文件读取、版本冻结和提示词配置，不是流程对象。
from curation.preparation.contracts import run_lock
from curation.preparation.notebook_io import freeze_run
from curation.preparation.config import DEFAULT
from curation.preparation.ops.image_filter import IMAGE_FILTER_DEFAULTS
from curation.preparation.image_filter_runtime import save_image_filter_policy

# view_saved只读已有结果；execute才执行下面唯一的pipeline入口。
# None 从原始数据开始；指定父run须匹配当前图片筛选policy。
MODE = 'view_saved'  # 只查看明确指定的最终结果，不新增模型调用
RUN = ROOT / 'curation/preparation/runs/pipeline_v2_knowledge_review'
from project import resolve_root
DATASET = resolve_root()  # 统一经 DEMIWTG_DATASETS_ROOT/配置解析；legacy 目录退役后不再指向仓库内旧路径
# 以下参数仅用于新建run。全文件处理：IDS=None、SAMPLE_RATE=1、MAX_RECORDS=None。
# 取消这些工程预算不等于全量吞吐及质量已经验收。
IDS = ['legacy:瓶式台球', 'legacy:高原', 'legacy:高锰酸钾', 'legacy:OK手势', 'legacy:白花芍药']  # 种子20260917随机3例 + 定向回归2例；抽样清单保存在旧双模型基线run
SOURCE_SCOPE = 'collected'  # 三个原始采集文件；all再包含QID与Wiki文件
SAMPLE_RATE, SEED, MAX_RECORDS, GROUP_SIZE = 1.0, 42, None, 256
THROUGH = 'export'  # 可改gather/identity/organize/extract/final_review/export
# 既有回归概念的明确身份范围；新概念使用概念记录，不按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
MODEL_CONFIG = {**IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'article_mode':True,'final_image_selection_only':True,'final_review_notes_required':True, 'final_draft_outline_only':False,'joint_thinking':True,'joint_reasoning_effort':'low','final_review_thinking':True,'final_review_effort':'medium','final_max_output_tokens':65536,'final_timeout_s':1200, 'final_input_tokens':131072, 'joint_input_tokens':32768, 'joint_image_target':4, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None, 'max_output_tokens':16384,
                'temperature':0, 'timeout_s':900, 'block_unit_chars':1800,
                'block_batch_chars':8000, 'comparison_group_chars':16000, 'image_batch_size':4,
                'text_embedding_model':str(ROOT.parent/'models/Qwen3-Embedding-0.6B'),
                'image_embedding_model':str(ROOT.parent/'models/siglip2-base-patch16-224')}  # 运行知识阶段时冻结本地模型配置与预算
# 修改代码/配置后执行须使用新的RUN目录，旧结果不可覆盖。

# 查看配置只影响展示，不影响处理或模型调用；新 run 实际结果在 Lance 中，按 stage_ref 固定版本读取。
FINAL_REF = None  # 完成后通过 stage_ref(RUN, 'knowledge_base') 获取固定引用
# view_saved绑定明确的新流程结果；质量状态见本册顶部及对应run/quality_review.json。
SAVED_RUN = RUN  # 指定当前 Lance run
VIEW = dict(concepts=None, limit=None, images=True)  # 只影响展示，不限制最终入选图片数量

from curation.preparation.ops.article import (PrepareArticleInput, ApplyArticle, PrepareFinalReview, PublishArticle, ArticleTokenBudget, PrepareSelectionScope, EnsureConceptLabel)












SAVED_REVIEW_RUNS = []

from curation.preparation.ops.visual_materials import ReuseImageAnnotations

from curation.preparation.visual_pipeline import load_graph as load_visual_graph, graph_hash as visual_graph_hash


## 2．完整 pipeline

以下函数是 notebook 与 CLI 共用的实际编排，可停在 gather、identity、organize、extract、final_review 或 export。

原始材料处理、筛选和容量分组后，模型阶段仅有联合提取与一次最终 review。最终输入超容量时明确停止，不截断；部分提取组失败时保留全部中间结果，阻止发布残缺概念文章。

当前默认仍选小批概念调试；200 概念 V1 历史批已完成，V2 需新 run，不续写旧批次。


In [ ]:
def run_pipeline(run, dataset, *, ids=None, sample_rate=1.0, seed=42, max_records_per_source=None, group_size=32, through='gather', model_config=None, project=ROOT, source_scope='collected'):
    """Dataset编排直接在这里；业务算子只处理行，不决定上下游。"""
    _assert_current_kernel()
    through = {'consolidate': 'final_review', 'fidelity': 'final_review', 'evidence': 'final_review'}.get(through, through)
    if through not in ['gather', 'identity', 'organize', 'extract', 'final_review', 'export']:
        raise ValueError('Unknown stopping stage')
    run, dataset = (Path(run), Path(dataset))
    if source_scope not in {'all', 'collected'}:
        raise ValueError('invalid source_scope')
    visual_graph = load_visual_graph()
    run_image_review = visual_graph['run_image_review']
    publish_visual_branch = visual_graph['publish_visual_branch']
    config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, **(model_config or {})}
    config.setdefault('image_annotations_ref', None)
    joint_concurrency = config.get('joint_concurrency', 1)
    if type(joint_concurrency) is not int or joint_concurrency < 1:
        raise ValueError('joint_concurrency must be a positive integer')
    final_review_concurrency = config.get('final_review_concurrency', 1)
    if type(final_review_concurrency) is not int or final_review_concurrency < 1:
        raise ValueError('final_review_concurrency must be a positive integer')
    global_audit = config.get('global_material_audit', False)
    settings = dict(pipeline_version='V2', visual_graph_sha256=visual_graph_hash(), ids=ids, sample_rate=sample_rate, seed=seed, max_records_per_source=max_records_per_source, group_size=group_size, model_config=config, source_scope=source_scope)
    with run_lock(run):
        tables = run / 'datasets'
        data = local_data()
        from curation.preparation.lake_inputs import resolve_source, read_source, DecodeSourceRow
        legacy_concepts_ref, legacy_concepts_source = resolve_source(dataset, 'legacy_concepts')
        collected_documents_ref, collected_documents_source = resolve_source(dataset, 'legacy_docs')
        collected_images_ref, collected_images_source = resolve_source(dataset, 'legacy_images')
        active_sources = [legacy_concepts_source, collected_documents_source, collected_images_source]
        if source_scope == 'all':
            qid_concepts_ref, qid_concepts_source = resolve_source(dataset, 'qid_concepts')
            wiki_ref, wiki_source = resolve_source(dataset, 'wiki_pages')
            active_sources += [qid_concepts_source, wiki_source]
        version = freeze_run(run, dataset, active_sources, settings, run_pipeline, project)
        legacy_concepts_records = data.read_lance(legacy_concepts_ref.resolve(dataset), version=legacy_concepts_ref.lance_version, limit=max_records_per_source).map(DecodeSourceRow(legacy_concepts_source)).filter(SelectSourceRecords('legacy_concepts', ids, sample_rate, seed, enabled=not global_audit)).map(EncodeStage('read_legacy_concepts')).checkpoint_lance(stage_uri(run, 'read_legacy_concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'read_legacy_concepts').map(from_stage_row)
        legacy_concepts = legacy_concepts_records.filter(lambda r: r['error'] is None and isinstance(r['value'], dict)).map(ConceptFromRecord(legacy_concepts_source)).map(EncodeStage('input_legacy_concepts')).checkpoint_lance(stage_uri(run, 'input_legacy_concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'input_legacy_concepts').map(from_stage_row)
        if source_scope == 'all':
            qid_concepts_records = read_source(data, dataset, qid_concepts_ref, qid_concepts_source, limit=max_records_per_source).filter(SelectSourceRecords('qid_concepts', ids, sample_rate, seed, enabled=not global_audit)).map(EncodeStage('read_qid_concepts')).checkpoint_lance(stage_uri(run, 'read_qid_concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'read_qid_concepts').map(from_stage_row)
            qid_concepts = qid_concepts_records.filter(lambda r: r['error'] is None and isinstance(r['value'], dict)).map(ConceptFromRecord(qid_concepts_source)).map(EncodeStage('input_qid_concepts')).checkpoint_lance(stage_uri(run, 'input_qid_concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'input_qid_concepts').map(from_stage_row)
        else:
            qid_concepts = data.from_iter(lambda: iter(()))
        collected_documents_records = read_source(data, dataset, collected_documents_ref, collected_documents_source, limit=max_records_per_source, ids=ids).filter(SelectSourceRecords('legacy_docs', ids, sample_rate, seed, enabled=not global_audit)).map(EncodeStage('read_collected_documents')).checkpoint_lance(stage_uri(run, 'read_collected_documents'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'read_collected_documents').map(from_stage_row)
        collected_documents = collected_documents_records.filter(lambda r: r['error'] is None and isinstance(r['value'], dict)).map(DocumentFromRecord(collected_documents_source)).map(EncodeStage('input_collected_documents')).checkpoint_lance(stage_uri(run, 'input_collected_documents'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'input_collected_documents').map(from_stage_row)
        collected_images_records = read_source(data, dataset, collected_images_ref, collected_images_source, limit=max_records_per_source, ids=ids).filter(SelectSourceRecords('legacy_images', ids, sample_rate, seed, enabled=not global_audit)).map(EncodeStage('read_collected_images')).checkpoint_lance(stage_uri(run, 'read_collected_images'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'read_collected_images').map(from_stage_row)
        collected_images = collected_images_records.filter(lambda r: r['error'] is None and isinstance(r['value'], dict)).map(ImageFromRecord()).map(EncodeStage('input_collected_images')).checkpoint_lance(stage_uri(run, 'input_collected_images'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'input_collected_images').map(from_stage_row)
        if source_scope == 'all':
            wiki_pages_records = read_source(data, dataset, wiki_ref, wiki_source, limit=max_records_per_source).map(EncodeStage('read_wiki_pages')).checkpoint_lance(stage_uri(run, 'read_wiki_pages'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'read_wiki_pages').map(from_stage_row)
            wiki_pages = wiki_pages_records.filter(lambda r: r['error'] is None and isinstance(r['value'], dict)).map(DocumentFromRecord()).map(EncodeStage('input_wiki_pages')).checkpoint_lance(stage_uri(run, 'input_wiki_pages'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'input_wiki_pages').map(from_stage_row)
        else:
            wiki_pages = data.from_iter(lambda: iter(()))
        concepts = legacy_concepts.union(qid_concepts).reduce_by_key('concept_ref', merge_concept).map(EncodeStage('concepts')).checkpoint_lance(stage_uri(run, 'concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'concepts').map(from_stage_row)
        images = collected_images.map(EncodeStage('images')).checkpoint_lance(stage_uri(run, 'images'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'images').map(from_stage_row)
        page_refs = concepts.flat_map(lambda c: c['page_refs']).reduce_by_key(['lang', 'page_id', 'mapped_concept_ref'], distinct).map(EncodeStage('page_refs')).checkpoint_lance(stage_uri(run, 'page_refs'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'page_refs').map(from_stage_row)
        wiki_documents = wiki_pages.join(page_refs, on=['lang', 'page_id'], how='left').reduce_by_key('doc_id', merge_document)
        documents = collected_documents.union(wiki_documents)
        documents = documents.map(EncodeStage('documents')).checkpoint_lance(stage_uri(run, 'documents'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'documents').map(from_stage_row)
        concept_selection = concepts.map(SelectConcept(ids, sample_rate, seed)).map(EncodeStage('concepts_selected')).checkpoint_lance(stage_uri(run, 'concepts_selected'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'concepts_selected').map(from_stage_row)
        selected_concepts = concept_selection.filter(lambda c: c['selected'])
        selected_keys = selected_concepts.select_columns(['concept_ref'])
        if ids is not None:
            data.from_iter(lambda: ({'concept_ref': ref} for ref in ids)).join(concepts.select_columns(['concept_ref']), on='concept_ref', how='anti').map(EncodeStage('missing_concepts')).checkpoint_lance(stage_uri(run, 'missing_concepts'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'missing_concepts').map(from_stage_row)
        all_document_links = documents.flat_map(MaterialLinks('doc_id'))
        all_image_links = images.flat_map(MaterialLinks('image_id'))
        document_links = all_document_links.join(selected_keys, on='concept_ref', how='semi').map(EncodeStage('documents_links')).checkpoint_lance(stage_uri(run, 'documents_links'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'documents_links').map(from_stage_row)
        image_links = all_image_links.join(selected_keys, on='concept_ref', how='semi').map(EncodeStage('images_links')).checkpoint_lance(stage_uri(run, 'images_links'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'images_links').map(from_stage_row)
        selected_documents = documents.join(document_links.select_columns(['doc_id']).reduce_by_key('doc_id', distinct), on='doc_id', how='semi').map(EncodeStage('documents_selected')).checkpoint_lance(stage_uri(run, 'documents_selected'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'documents_selected').map(from_stage_row)
        selected_images = images.join(image_links.select_columns(['image_id']).reduce_by_key('image_id', distinct), on='image_id', how='semi').map(EncodeStage('images_selected')).checkpoint_lance(stage_uri(run, 'images_selected'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'images_selected').map(from_stage_row)
        if global_audit:
            for objects, links, key, name in [(documents, all_document_links, 'doc_id', 'documents'), (images, all_image_links, 'image_id', 'images')]:
                associated = links.join(concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').select_columns([key]).reduce_by_key(key, distinct)
                objects.join(associated, on=key, how='anti').map(EncodeStage(f'{name}_unmatched')).checkpoint_lance(stage_uri(run, f'{name}_unmatched'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + f'{name}_unmatched').map(from_stage_row)
        processed_documents = selected_documents.map_async(ReadDocument(dataset)).map_async(CleanDocument()).map(FilterDocumentBlocks()).map(EncodeStage('documents_processed')).checkpoint_lance(stage_uri(run, 'documents_processed'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'documents_processed').map(from_stage_row)
        processed_images = selected_images.map_async(CheckImage(dataset)).map(EncodeStage('images_processed')).checkpoint_lance(stage_uri(run, 'images_processed'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'images_processed').map(from_stage_row)
        processed_images = processed_images.map(ReuseImageAnnotations(config.get('image_annotations_ref'), config.get('image_annotation_config_id'))).map(EncodeStage('images_indexed')).checkpoint_lance(stage_uri(run, 'images_indexed'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'images_indexed').map(from_stage_row)
        document_counts = document_links.join(processed_documents.select_columns(['doc_id', 'read_status']), on='doc_id').reduce_by_key('concept_ref', CountMaterial('document_count', 'read_status', 'readable_documents'))
        image_counts = image_links.join(processed_images.select_columns(['image_id', 'byte_status']), on='image_id').reduce_by_key('concept_ref', CountMaterial('image_count', 'byte_status', 'verified_images'))
        concepts_ready = selected_concepts.join(document_counts, on='concept_ref', how='left').join(image_counts, on='concept_ref', how='left').map(fill_material_counts).map(EncodeStage('concepts_ready')).checkpoint_lance(stage_uri(run, 'concepts_ready'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'concepts_ready').map(from_stage_row)
        concept_documents = document_links.join(processed_documents.map(NestMaterial('doc_id', 'documents')), on='doc_id')
        concept_images = image_links.join(processed_images.map(NestMaterial('image_id', 'images')), on='image_id')
        material_batches = concept_documents.union(concept_images).group_batches('concept_ref', max_rows=group_size, output='materials')
        batches = concepts_ready.join(material_batches, on='concept_ref', how='left').map(EncodeStage('knowledge_inputs')).checkpoint_lance(stage_uri(run, 'knowledge_inputs'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'knowledge_inputs').map(from_stage_row)
        if through == 'gather':
            return batches
        knowledge_run = run / 'knowledge'
        pack, prompt_text = material_prompt_pack(config)
        options = prompt_execution_options(run, config)
        save_prompt_config(run, prompt_text, options)
        prompt_data = local_data(prompt_packs={'knowledge.yaml': pack}, max_prompt_requests=config['max_calls'], prompt_options=options)
        batches = read_stage(run, 'knowledge_inputs', data=prompt_data)
        identified = batches.map(model_input).map_async(PrepareIdentity(knowledge_run, config)).map_prompt_async('identity', config='knowledge.yaml', inputs={'payload': 'identity_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', when=lambda r: not r.get('blocked') and 'identity_prompt' in r, concurrency=1, queue_depth=1).map_async(ApplyIdentity(knowledge_run, config)).map(EncodeStage('knowledge_identity')).checkpoint_lance(stage_uri(run, 'knowledge_identity'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'knowledge_identity').map(from_stage_row)
        if through == 'identity':
            return identified
        if config.get('text_mode') == 'multimodal':
            blocks = identified.map(EnsureConceptLabel()).map(BuildSourceBlocks(config.get('block_unit_chars', 1800), body_only=True)).map(SelectAvailableImages()).map(EncodeStage('multimodal_materials')).checkpoint_lance(stage_uri(run, 'multimodal_materials'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'multimodal_materials').map(from_stage_row)
            text_requests = blocks.flat_map(BatchSourceBlocks(config.get('block_batch_chars', 8000))).map(PrepareSelectionScope(config['image_identity_definitions']))
            text_decisions = text_requests.map_prompt_async('select_blocks', config='knowledge.yaml', inputs={'payload': 'block_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).map(ApplyBlockSelection(relevance_only=True)).map(EncodeStage('text_relevance')).checkpoint_lance(stage_uri(run, 'text_relevance'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'text_relevance').map(from_stage_row).reduce_by_key('case_id', merge_block_decisions)
            image_decisions = run_image_review(blocks, run, config, version)
            related = blocks.join(text_decisions, on='case_id', how='left').join(image_decisions, on='case_id', how='left').map(SelectRelatedMaterials()).map(EncodeStage('related_materials')).checkpoint_lance(stage_uri(run, 'related_materials'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'related_materials').map(from_stage_row)
            save_image_filter_policy(run, config)
            related = publish_visual_branch(related, run, version)
            if through == 'organize':
                return related
            routing_materials = related.map(PrepareRoutingMaterials()).map(EncodeStage('routing_materials')).checkpoint_lance(stage_uri(run, 'routing_materials'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'routing_materials').map(from_stage_row)
            text_embeddings = routing_materials.flat_map(RawPassageRows()).group_batches('embedding_bucket', max_rows=2, output='items').map(EmbedParagraphBatch(config['text_embedding_model'])).map(EncodeStage('material_text_embeddings')).checkpoint_lance(stage_uri(run, 'material_text_embeddings'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'material_text_embeddings').map(from_stage_row).flat_map(lambda r: r['items']).reduce_by_key('case_id', lambda acc, r: {'case_id': r['case_id'], 'passage_embeddings': {**acc['passage_embeddings'], r['source_id']: r}}, initial={'passage_embeddings': {}})
            image_text_embeddings = routing_materials.map(EncodeImageTextMaterials(config['image_embedding_model'])).map(EncodeStage('material_image_embeddings')).checkpoint_lance(stage_uri(run, 'material_image_embeddings'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'material_image_embeddings').map(from_stage_row)
            routed = routing_materials.join(text_embeddings, on='case_id', how='left').join(image_text_embeddings.select_columns(['case_id', 'text_windows', 'image_vectors']), on='case_id', how='left').map(RouteByTokenBudget(ROOT.parent / 'models/Qwen3.8-27B', config.get('joint_input_tokens', 32768), counter=ArticleTokenBudget(ROOT.parent / 'models/Qwen3.8-27B', {**config, 'enable_thinking': config.get('joint_thinking', True), 'reasoning_effort': config.get('joint_reasoning_effort', 'low')}, 'joint_paragraphs'), max_images=config.get('joint_image_target', 4))).map(EncodeStage('material_routing')).checkpoint_lance(stage_uri(run, 'material_routing'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'material_routing').map(from_stage_row)
            joint_requests = routed.flat_map(lambda r: r['requests']).map(BuildRoutedJointRequest()).map(EncodeStage('joint_requests')).checkpoint_lance(stage_uri(run, 'joint_requests'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'joint_requests').map(from_stage_row)
            joint_config = {**config, 'enable_thinking': config.get('joint_thinking', True), 'reasoning_effort': config.get('joint_reasoning_effort', 'low')}
            joint_options = prompt_execution_options(run, joint_config)
            save_prompt_config(run / 'joint_extraction', prompt_text, joint_options)
            joint_data = local_data(prompt_packs={'knowledge.yaml': pack}, prompt_options=joint_options, max_prompt_requests=config['max_calls'])
            article_inputs = read_stage(run, 'joint_requests', data=joint_data).map(PrepareArticleInput(config['image_identity_definitions'])).map(EncodeStage('article_inputs')).checkpoint_lance(stage_uri(run, 'article_inputs'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'article_inputs').map(from_stage_row)
            extracted = article_inputs.map_prompt_async('joint_paragraphs', config='knowledge.yaml', inputs={'payload': 'article_input', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=joint_concurrency, queue_depth=joint_concurrency).map(ApplyArticle()).map(EncodeStage('paragraph_extract')).checkpoint_lance(stage_uri(run, 'paragraph_extract'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'paragraph_extract').map(from_stage_row)
            if through == 'extract':
                return extracted
            draft_groups = extracted.reduce_by_key('concept', lambda acc, r: {'concept': r['concept'], 'drafts': acc['drafts'] + [r]}, initial={'drafts': []})
            review_config = {**config, 'enable_thinking': config.get('final_review_thinking', True), 'reasoning_effort': config.get('final_review_effort', 'low'), 'max_output_tokens': config.get('final_max_output_tokens', 32768), 'timeout_s': config.get('final_timeout_s', 1200)}
            review_options = prompt_execution_options(run, review_config)
            save_prompt_config(run / 'final_review', prompt_text, review_options)
            review_data = local_data(prompt_packs={'knowledge.yaml': pack}, prompt_options=review_options, max_prompt_requests=config['max_calls'])
            review_requests = draft_groups.map(PrepareFinalReview(config['image_identity_definitions'], ArticleTokenBudget(ROOT.parent / 'models/Qwen3.8-27B', review_config, 'final_review'), config.get('final_input_tokens', 131072), draft_outline_only=config.get('final_draft_outline_only', False))).map(EncodeStage('final_review_requests')).checkpoint_lance(stage_uri(run, 'final_review_requests'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'final_review_requests').map(from_stage_row)
            reviewed = read_stage(run, 'final_review_requests', data=review_data).map_prompt_async('final_review', config='knowledge.yaml', inputs={'payload': 'article_input', 'images': 'pixel_images'}, when=lambda r: not r['preflight_error'], output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=final_review_concurrency, queue_depth=final_review_concurrency).map(ApplyArticle(final=True, image_selection_only=config.get('final_image_selection_only', False), review_notes_required=config.get('final_review_notes_required', False))).map(EncodeStage('final_review')).checkpoint_lance(stage_uri(run, 'final_review'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'final_review').map(from_stage_row)
            if through == 'final_review':
                return reviewed
            material_groups = related.map(lambda r: {'concept': r['identity']['target_label'], 'material': r}).reduce_by_key('concept', lambda acc, r: {'concept': r['concept'], 'materials': acc['materials'] + [r['material']]}, initial={'materials': []})
            result = material_groups.join(reviewed.map(lambda r: {'concept': r['concept'], 'review': r}), on='concept', how='left').map(PublishArticle()).map(EncodeStage('knowledge_base')).checkpoint_lance(stage_uri(run, 'knowledge_base'), schema=PIPELINE_STAGE_ROWS, fingerprint=version + ':' + 'knowledge_base').map(from_stage_row)
            from curation.preparation.publication import publish_entity_stage
            publish_entity_stage(run, 'knowledge')
            return result
        raise ValueError('Formal notebook supports text_mode=multimodal only')


## 3．pipeline 最终结果

execute 模式展示 RUN，view_saved 模式展示明确指定的 SAVED_RUN；无结果时显示缺失，绝不自动回退。默认展示全部最终主题、段落、图片及其字段、引用；处理状态明确显示，完整材料与过程记录可通过 audit=True 展开。


In [ ]:
import importlib
from curation.preparation import current_results
importlib.reload(current_results)  # 仅刷新展示模块，不热加载业务算子
from curation.preparation.current_results import show_current_results
show_current_results(RUN if MODE == 'execute' else SAVED_RUN, **VIEW)
if MODE == 'view_saved':
    for saved_review in SAVED_REVIEW_RUNS:
        show_current_results(saved_review, **VIEW)
